In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import pandas as pd

In [18]:
# Import prepared data
df = pd.read_csv('../data/processed/telco_customer_churn_data_processed.csv')

We will want to compare our different models to choose the best performing model:

In [19]:
accuracy_list = []

In [20]:
# Prepare the data
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Our significant metric for churn analysis is "Recall". Recall is calculated using the formula:
$$
\text{Recall} = \frac{TP}{TP + FN}
$$
where:
- TP = True Positive (predicting: churn, truth: churn)
- FN = False Negative (predicting: not-churn, truth: churn)

We focus on recall as we want to minimize the number of customers who leave, and if we accidentally identify a loyal customer as a churner, it's not a big deal - we just want to ensure the company doesn't potentially lose a customer, even if it might be an unnecessary investment.

For this reason, during the classification report we will be focusing on the recall value for class 1 (churners) to choose our best model.

1. Random Forest Classifier

In [21]:
from sklearn.metrics import classification_report

# Train random forest classifier
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

THRESHOLD = 0.1

# Evaluate the model
prob = rf_model.predict_proba(X_test)[:, 1]
y_pred = (prob >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.942     0.490     0.645      1035
           1      0.394     0.917     0.551       374

    accuracy                          0.603      1409
   macro avg      0.668     0.703     0.598      1409
weighted avg      0.797     0.603     0.620      1409



In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = rf_model.predict_proba(X_test)[:, 1]

print("Threshold tuning for Random Forest:")
for threshold in [0.10, 0.30, 0.50, 0.70, 0.90]:
    y_pred = (proba >= threshold).astype(int)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print(f"Threshold: {threshold}, Precision: {precision:.3f}, Recall: {recall:.3f}, F1-Score: {f1:.3f}")

Threshold tuning for Random Forest:
Threshold: 0.1, Precision: 0.394, Recall: 0.917, F1-Score: 0.551
Threshold: 0.3, Precision: 0.531, Recall: 0.725, F1-Score: 0.613
Threshold: 0.5, Precision: 0.638, Recall: 0.500, F1-Score: 0.561
Threshold: 0.7, Precision: 0.736, Recall: 0.254, F1-Score: 0.378
Threshold: 0.9, Precision: 0.839, Recall: 0.070, F1-Score: 0.128


Light GBM Classifier

In [23]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
import time

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

# Training timer
start_time = time.time()
lgbm_model.fit(X_train, y_train)
end_time = time.time()
train_time = end_time - start_time
print(f'LightGBM Training Time: {train_time:.2f} seconds')

# Prediction timer
start_time = time.time()
proba = lgbm_model.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)
end_time = time.time()
pred_time = end_time - start_time
print(f'LightGBM Prediction Time: {pred_time:.2f} seconds')

print(classification_report(y_test, y_pred, digits=3))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000431 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 627
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
LightGBM Training Time: 0.30 seconds
LightGBM Prediction Time: 0.00 seconds
              precision    recall  f1-score   support

           0      0.948     0.475     0.633      1035
           1      0.390     0.928     0.549       374

    accuracy                          0.595      1409
   macro avg      0.669     0.702     0.591      1409
weighted avg      0.

XGBoost Decision Tree

In [24]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight for XGBoost to handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.961     0.448     0.611      1035
           1      0.383     0.949     0.546       374

    accuracy                          0.581      1409
   macro avg      0.672     0.699     0.579      1409
weighted avg      0.807     0.581     0.594      1409

